# One classifier trained across all three ZuCo tasks

**T4 GPU → Run all.** Use your **Zyion Drive account** when mounting and authenticating Drive.

- Binary **positive versus negative**; combine SR + NR + TSR into one training pool. Each cross-validation fold trains one shared model.
- **CNN:** all 105 channels, 0–792 ms, compact EEGNet-inspired model. **Logistic regression:** all channels averaged over 224–304 ms, the window examined in earlier work. This is a comparison of complete pipelines, not a controlled architecture-only comparison.
- Test each model on **individual trials** and **same-class averages of up to 40 trials**. Keep smaller final groups to retain scarce NR/TSR data. Their actual sizes are reported.
- Three separate evaluations: **unseen sentences**, **unseen word identities**, **unseen readers**. Only the named grouping is excluded; the other two may overlap. Task identities always overlap.
- Saved **40 Hz / 125 Hz first-stored-fixation EEG archive** and original word-agreement labels. No new extraction, relabeling or paper regression cleaning. This is separate from the later reconstructed paper-style SR trial set.

**Storage:** at most **100 MB new Drive outputs**. Only the selected saved EEG is copied to temporary Colab disk (roughly 0.5 GB). Loading it takes time. Completed fits are reused; restarting the runtime requires reloading the EEG, not re-extracting it. Training resumes from the last saved checkpoint, every five epochs or at early stopping.

**Interpretation:** test averages use the known class labels to group trials. Their scores describe class separation after averaging, not accuracy for one unknown word. Balanced-accuracy chance is 50%. A task-only control checks whether task differences alone can predict the labels.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'GPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


## 1. Connect and verify the saved inputs

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/zuco_pooled_cnn'
REPO=Path('/content/word-sentiment-eeg-pooled')
CODE_REVISION='f1da6e4abc6da07223c1e4ecf3576c91a8a17cf0'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-analysis.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

import torch
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot access the GPU. Reconnect using T4 GPU, then Run all.')
print('Training GPU:',torch.cuda.get_device_name(0))

# Check the authenticated account quota, not the mounted runtime volume.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
session=AuthorizedSession(credentials)
response=session.get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated Drive account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication. This notebook expects its existing EEG archive.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify available space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,100_000_000-used):
    raise RuntimeError('Reserve up to 100 MB for new outputs. No EEG preparation has started.')
print(f'Free Drive space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 100 MB.')
print('Saved results:',OUT)


## 2. Load saved EEG and run all comparisons

Progress shows reader/task recording **out of 36**, shard progress, model, fold and epoch. No GPU training happens on this Mac. Three folds per holdout, grouped validation, train-only scaling and class-weighted training; no test-score tuning. Word holdouts additionally remove overlapping training EEG windows within the same reader/task/sentence.

In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/classify_zuco_pooled.py'),
         '--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='Pooled ZuCo CNN and averaged logistic classification')


## 3. Input counts and run completion

These are accepted word–reader EEG instances, summed across readers. No multiplication by reader count is needed.

In [ ]:
completion=json.loads((OUT/'complete.json').read_text())
print('Run:',completion['status'],'| Completed cases:',completion['cases'])
counts=json.loads((OUT/'input_counts.json').read_text())
display(pd.DataFrame(counts['task_class_counts']).T.rename_axis('task'))
print('Total selected EEG instances:',counts['selected_trials'],'| Readers:',counts['readers'])
print('Duplicate observations excluded before binary selection:',counts['duplicate_observations_excluded'])


## 4. Classification results

**Pooled BA:** class-balanced score over all test instances. **Equal-task BA:** mean of the three task scores, so SR cannot dominate by contributing more trials. Per-task scores below are predictions from the same pooled model, not separately trained models. All scores are out-of-fold; they are descriptive estimates without permutation-based significance tests.

In [ ]:
scores=pd.read_csv(OUT/'scores.csv')
view=scores.copy()
for col in ['pooled_ba','equal_task_ba','SR_ba','NR_ba','TSR_ba']:
    view[col]=(view[col]*100).round(1)
display(view.rename(columns={'max_trials_per_average':'trials per average (maximum)'}))
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(17,4),sharey=True)
colors={'cnn':'#2874a6','logistic':'#148f77','dummy':'#888888','task_only':'#ba6b25'}
for ax,split in zip(axes,['sentence','word','reader']):
    for model in ['cnn','logistic','dummy','task_only']:
        part=scores[(scores.split==split)&(scores.model==model)].sort_values('max_trials_per_average')
        ax.plot(part.max_trials_per_average,100*part.equal_task_ba,'o-',label=model,color=colors[model])
    ax.axhline(50,color='black',ls='--',lw=1)
    ax.set(title='Unseen '+split+'s',xlabel='Maximum trials per same-class average',xticks=[1,40])
axes[0].set_ylabel('Equal-task balanced accuracy (%)')
axes[-1].legend();fig.tight_layout();display(fig);plt.close(fig)


## 5. Average sizes and reader-level checks

Smaller remainder groups are retained; an “up to 40” example can contain fewer than 40 trials. This differs from the earlier experiment that discarded remainders. Reader rows below are subgroup evaluations of pooled models, not subject-specific training.

In [ ]:
results=json.loads((OUT/'summary.json').read_text())
display(pd.DataFrame([dict(split=r['split'],model=r['model'],maximum=r['max_trials_per_average'],**r['average_sizes']) for r in results]))
reader_rows=[]
for r in results:
    if r['model'] in ['cnn','logistic']:
        for reader,row in r['per_reader'].items():
            reader_rows.append(dict(split=r['split'],model=r['model'],maximum=r['max_trials_per_average'],reader=reader,**row))
display(pd.DataFrame(reader_rows).drop(columns=['confusion'],errors='ignore'))
print('Saved tables, predictions, splits, protocol and small model checkpoints:',OUT)
print('Primary comparison: CNN and logistic versus both controls, with equal-task and per-task scores.')
